In [1]:
!nvidia-smi

Sat Sep  5 05:53:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip uninstall -y transformers
!pip install -q git+https://github.com/huggingface/transformers.git@096f25ae1f501a084d8ff2dcaf25fbc2bd60eba4

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu128
Transformers: 4.52.0.dev0
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading BitNet model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
)

model = model.to("cuda")

print("✅ BitNet model loaded successfully!")

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

You have loaded a BitNet model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


model.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

✅ BitNet model loaded successfully!


In [5]:
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

Model device: cuda:0
CUDA available: True


In [6]:
import time
import torch

def test_bitnet(prompt, max_new_tokens=100):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # Reset GPU memory measurement
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    # Start timing
    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    # Wait for GPU to finish
    torch.cuda.synchronize()

    end_time = time.perf_counter()

    # Calculate metrics
    elapsed_time = end_time - start_time

    input_tokens = inputs["input_ids"].shape[1]
    output_tokens = outputs.shape[1] - input_tokens

    tokens_per_second = output_tokens / elapsed_time

    peak_memory = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    # Decode response
    response = tokenizer.decode(
        outputs[0][input_tokens:],
        skip_special_tokens=True
    )

    print("=" * 60)
    print("PROMPT")
    print("=" * 60)
    print(prompt)

    print("\n" + "=" * 60)
    print("RESPONSE")
    print("=" * 60)
    print(response)

    print("\n" + "=" * 60)
    print("PERFORMANCE")
    print("=" * 60)

    print(f"Generation time : {elapsed_time:.2f} seconds")
    print(f"Output tokens   : {output_tokens}")
    print(f"Tokens/second   : {tokens_per_second:.2f}")
    print(f"Peak GPU memory : {peak_memory:.2f} GB")

    return {
        "prompt": prompt,
        "response": response,
        "generation_time": elapsed_time,
        "output_tokens": output_tokens,
        "tokens_per_second": tokens_per_second,
        "peak_gpu_memory": peak_memory
    }

In [7]:
result = test_bitnet(
    "Explain what a 1-bit large language model is in simple terms."
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:2941: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


PROMPT
Explain what a 1-bit large language model is in simple terms.

RESPONSE
A 1-bit large language model is a simplified version of a large language model, which is a type of artificial intelligence that can understand and generate human language. In this case, the "1-bit" part means that the model is only able to represent a single bit of information, which is a very basic unit of data.

This is not a very accurate or useful representation of a language model, but it can be thought of as a very basic and limited version of a more complex language model.

PERFORMANCE
Generation time : 17.19 seconds
Output tokens   : 100
Tokens/second   : 5.82
Peak GPU memory : 4.54 GB


In [8]:
test_prompts = [
    "Explain artificial intelligence in simple terms.",

    "Write a Python function to check whether a number is prime.",

    "What is the difference between 1-bit, 4-bit and 16-bit model weights?",

    "A shop has 5 apples. It sells 2 apples and buys 4 more. How many apples does it have?",

    "Give me exactly 5 benefits of running an LLM locally.",

    "Write a professional email asking a manager for a project status update.",

    "Explain how an LLM generates a response from a user prompt."
]

In [9]:
results = []

for prompt in test_prompts:

    result = test_bitnet(
        prompt,
        max_new_tokens=100
    )

    results.append(result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial intelligence, or AI, is like a super-smart computer program that can learn and make decisions just like a human. It's like having a really, really smart robot that can do lots of things on its own, like understanding language, recognizing images, and even playing games.

Think of it like a really smart student who can answer questions, solve problems, and even learn new things. It's not just about following simple instructions, but about understanding and reasoning about the world around it.

PERFORMANCE
Generation time : 17.72 seconds
Output tokens   : 100
Tokens/second   : 5.64
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Write a Python function to check whether a number is prime.

RESPONSE
Here's a simple Python function to check if a number is prime:

```python
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
```

This function works by checking if the number has any divisors other than 1 and itself. If it finds a divisor, it immediately returns False

PERFORMANCE
Generation time : 13.33 seconds
Output tokens   : 100
Tokens/second   : 7.50
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

PROMPT
A shop has 5 apples. It sells 2 apples and buys 4 more. How many apples does it have?

RESPONSE
To find out how many apples the shop has, you need to follow these steps:

1. Start with the initial number of apples: 5 apples.
2. Subtract the number of apples sold: 5 apples - 2 apples = 3 apples.
3. Add the number of apples bought: 3 apples + 4 apples = 7 apples.

So, the shop has 7 apples.

PERFORMANCE
Generation time : 10.44 seconds
Output tokens   : 83
Tokens/second   : 7.95
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Give me exactly 5 benefits of running an LLM locally.

RESPONSE
1. **Data Privacy**: Running an LLM locally allows you to keep your data secure and private, as it is not stored in a centralized cloud service.
2. **Reduced Latency**: Processing data locally can reduce latency, making real-time applications more responsive and efficient.
3. **Cost-Effective**: It can be more cost-effective than using cloud services for large-scale operations, as you only pay for the resources you use.
4. **Customization**: Local deployment enables you to tailor

PERFORMANCE
Generation time : 12.10 seconds
Output tokens   : 100
Tokens/second   : 8.26
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Write a professional email asking a manager for a project status update.

RESPONSE
Subject: Request for Project Status Update

Dear [Manager's Name],

I hope this email finds you well. I am writing to request an update on the [Project Name] project. As we are currently in the middle of our project timeline, I would appreciate it if you could provide me with the current status of the project, including any key milestones achieved, upcoming deadlines, and any potential roadblocks or challenges that may impact our timeline.

Additionally, I would like to know if there are any changes

PERFORMANCE
Generation time : 12.19 seconds
Output tokens   : 100
Tokens/second   : 8.21
Peak GPU memory : 4.54 GB
PROMPT
Explain how an LLM generates a response from a user prompt.

RESPONSE
Sure! Here's an overview of how a Large Language Model (LLM) like me generates a response to a user prompt:

1. **Input Parsing**: When you input a query, it is first broken down into individual words or tokens. 

In [10]:
results_200 = []

for prompt in test_prompts:
    result = test_bitnet(
        prompt,
        max_new_tokens=200
    )
    results_200.append(result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial intelligence, or AI, is like a smart computer program that can learn from data and make decisions on its own. It's like having a really smart assistant that can help you with tasks, answer questions, and even predict what you might want to do next. AI can be used in many different ways, like helping doctors diagnose diseases, driving cars, or even playing games. It's like having a super-smart friend that can help you do things better and faster.

PERFORMANCE
Generation time : 13.11 seconds
Output tokens   : 97
Tokens/second   : 7.40
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Write a Python function to check whether a number is prime.

RESPONSE
```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
```

This function works by checking divisibility of the number `n` from 2 to the square root of `n`. If any number in this range divides `n` evenly, then `n` is not prime. If no such number is found, `n` is prime. 

You can use this function like this:

```python
print(is_prime(11))  # True
print(is_prime(15))  # False
```

PERFORMANCE
Generation time : 18.51 seconds
Output tokens   : 148
Tokens/second   : 7.99
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

PROMPT
A shop has 5 apples. It sells 2 apples and buys 4 more. How many apples does it have?

RESPONSE
To find out how many apples the shop has, we need to consider both the apples sold and the apples bought.

Initially, the shop has 5 apples.

The shop sells 2 apples, so we subtract 2 from 5:
5 - 2 = 3

Then, the shop buys 4 more apples, so we add 4 to 3:
3 + 4 = 7

So, the shop now has 7 apples.

PERFORMANCE
Generation time : 11.71 seconds
Output tokens   : 94
Tokens/second   : 8.03
Peak GPU memory : 4.54 GB


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Give me exactly 5 benefits of running an LLM locally.

RESPONSE
1. **Data Privacy**: Running an LLM locally allows you to keep data within your organization, ensuring better control over sensitive information and compliance with data protection regulations.

2. **Reduced Latency**: Processing data locally can significantly reduce latency compared to sending it to a remote server, which can be crucial for real-time applications.

3. **Cost Efficiency**: Local computation can save on cloud costs, as you don't need to pay for expensive cloud resources for running the model.

4. **Customization**: Running a model locally enables you to fine-tune it on your specific data, which can improve its performance and accuracy for your particular use case.

5. **Security**: Having a local model reduces the risk of data breaches and cyber attacks, as the data is not stored in the cloud.

PERFORMANCE
Generation time : 18.88 seconds
Output tokens   : 161
Tokens/second   : 8.53
Peak GPU memory : 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Write a professional email asking a manager for a project status update.

RESPONSE
Subject: Request for Project Status Update

Dear [Manager's Name],

I hope this message finds you well.

I am writing to request an update on the [Project Name] project. As we approach the project deadline, I would appreciate it if you could provide an overview of the current status, including any recent developments, progress, and any potential challenges that may arise.

Additionally, I would like to discuss the next steps and ensure that we are on track to meet our objectives. Please let me know if there are any specific areas you would like to focus on or if there are any additional details you would like me to be aware of.

Thank you for your attention to this matter. I look forward to your response.

Best regards,

[Your Name]  
[Your Position]  
[Your Contact Information]

PERFORMANCE
Generation time : 20.71 seconds
Output tokens   : 163
Tokens/second   : 7.87
Peak GPU memory : 4.55 GB
PROM

In [11]:
prompt = "Explain artificial intelligence in simple terms."

runs = []

for i in range(5):
    print(f"\n========== Run {i+1} ==========")

    result = test_bitnet(
        prompt,
        max_new_tokens=100
    )

    runs.append(result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



========== Run 1 ==========


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial intelligence (AI) is a type of technology that allows computers to perform tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, and making decisions. It's like giving a computer the ability to think, learn, and solve problems in a way that's similar to how humans do.

Imagine a computer program that can understand what you're saying, recognize images, and even make decisions based on data. That's AI in action! It's used in many areas, such

PERFORMANCE
Generation time : 15.24 seconds
Output tokens   : 100
Tokens/second   : 6.56
Peak GPU memory : 4.54 GB

========== Run 2 ==========


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial Intelligence, or AI, is like teaching a computer to think and learn like a human. It's a way for machines to process information, make decisions, and solve problems without being explicitly programmed. AI can help computers understand language, recognize patterns, and even make predictions based on data.

Think of it like a smart assistant that can help you find information, answer questions, and make decisions. It's like having a super-smart friend who can help you with lots of tasks and tasks that

PERFORMANCE
Generation time : 12.71 seconds
Output tokens   : 100
Tokens/second   : 7.87
Peak GPU memory : 4.54 GB

========== Run 3 ==========


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial intelligence, or AI, is like a super-smart computer program that can learn from data, just like humans do. It can help us solve problems, make decisions, and even do some tasks on our own. It's like having a really smart friend who can help us understand things and make sense of the world around us. AI can be used in many ways, like helping doctors diagnose diseases, recommending movies you might like, or even driving cars. It's a really cool technology that

PERFORMANCE
Generation time : 12.86 seconds
Output tokens   : 100
Tokens/second   : 7.77
Peak GPU memory : 4.54 GB

========== Run 4 ==========


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial intelligence, or AI, is like a smart computer program that can learn and make decisions on its own. It's like having a super-smart friend who can help you with tasks, answer questions, and even predict what you might want to do next.

Think of it like a robot that can understand and respond to your words, like Siri or Alexa. These programs use complex math and special algorithms to process information and figure out what to do next. They can recognize patterns, make decisions,

PERFORMANCE
Generation time : 13.06 seconds
Output tokens   : 100
Tokens/second   : 7.66
Peak GPU memory : 4.54 GB

========== Run 5 ==========
PROMPT
Explain artificial intelligence in simple terms.

RESPONSE
Artificial intelligence, or AI, is like a super-smart computer program that can learn from experience, recognize patterns, and make decisions on its own. It's a bit like having a really, really smart friend who can help you solve 

In [12]:
times = [r["generation_time"] for r in runs]
tokens_sec = [r["tokens_per_second"] for r in runs]
memory = [r["peak_gpu_memory"] for r in runs]

print("\n========== AVERAGE ==========")
print(f"Average generation time : {sum(times)/len(times):.2f} sec")
print(f"Average tokens/sec      : {sum(tokens_sec)/len(tokens_sec):.2f}")
print(f"Average peak GPU memory : {sum(memory)/len(memory):.2f} GB")


========== AVERAGE ==========
Average generation time : 13.34 sec
Average tokens/sec      : 7.53
Average peak GPU memory : 4.54 GB
